# Shopper Insights ML — Complete Exploratory Data Analysis
**Dataset:** UCI Online Shoppers Purchasing Intention (ID 468)  
**Goal:** Understand data quality, class imbalance, feature distributions, relationships, and preprocessing needs before training five classifiers.  
**How to use:** Open in VS Code, select your `.venv` kernel, and run cells top-to-bottom. Place `online_shoppers_intention.csv` in `data/raw/`. This notebook does **not** train the assignment models or modify the original CSV.

## 0. Imports and dataset loading
Run from either the project root or `notebooks/` folder. Install dependencies with `python -m pip install pandas numpy matplotlib seaborn scikit-learn ipykernel`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

candidates = [Path("data/raw/online_shoppers_intention.csv"), Path("../data/raw/online_shoppers_intention.csv")]
found = [p for p in candidates if p.exists()]
if not found:
    raise FileNotFoundError("Place online_shoppers_intention.csv in data/raw/ and run from the project root or notebooks/.")
DATA_PATH = found[0].resolve()
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 1. Structure and descriptive statistics
There should be **17 predictors and 1 target (`Revenue`)**. Verify this against your own file; do not assume every downloaded copy is identical.

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nNumerical summary:")
display(df.describe())
print("\nAll-column summary:")
display(df.describe(include="all"))
assert "Revenue" in df.columns, "Missing Revenue target column"
assert df.shape[0] >= 500 and df.shape[1] - 1 >= 12, "Dataset does not meet assignment minimums"

## 2. Missing values and duplicate analysis
`isna().sum()` counts missing entries per column. `duplicated()` flags repeated *complete rows*. `keep=False` displays all members of each repeated group. Do not delete duplicates automatically: identical feature values may represent separate sessions.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (100 * df.isna().mean()).round(2)
}).sort_values("Missing Count", ascending=False)
display(missing_summary)
print("Total missing cells:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate row percentage:", round(100 * df.duplicated().mean(), 2))
display(df.loc[df.duplicated(keep=False)].head(20))

In [ ]:
plt.figure(figsize=(13, 4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="Blues")
plt.title("Missing-value map (uniform if no values are missing)")
plt.tight_layout()
plt.show()

feature_cols = df.columns.drop("Revenue").tolist()
conflicting = df.groupby(feature_cols, dropna=False)["Revenue"].nunique(dropna=False)
conflicting_count = int((conflicting > 1).sum())
quality_report = pd.DataFrame({
    "Check": ["Records", "Predictors", "Missing cells", "Missing targets", "Exact duplicate rows", "Conflicting-label feature groups"],
    "Result": [len(df), len(feature_cols), int(df.isna().sum().sum()), int(df["Revenue"].isna().sum()), int(df.duplicated().sum()), conflicting_count]
})
display(quality_report)

## 3. Target class distribution and baseline
The original UCI data has 10,422 non-purchasing and 1,908 purchasing sessions (~84.53% vs 15.47%). **Use your computed results** if the local CSV differs. A majority-only predictor has high accuracy but zero purchase recall.

In [ ]:
# Supports booleans, 0/1, and string "True"/"False" target encodings.
y = df["Revenue"].astype(str).str.strip().str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})
if y.isna().any():
    raise ValueError("Revenue has unexpected or missing labels; inspect df['Revenue'].unique()")
y = y.astype(int)
counts = y.value_counts().reindex([0, 1], fill_value=0)
class_summary = pd.DataFrame({"Count": counts, "Percentage": (counts / len(y) * 100).round(2)})
class_summary.index = ["No Purchase (0)", "Purchase (1)"]
display(class_summary)
print("Majority/minority ratio:", round(counts.max() / counts.min(), 2) if counts.min() else "Undefined")
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=["No Purchase", "Purchase"], y=counts.values, color="#4361ee", ax=ax)
for i, value in enumerate(counts.values):
    ax.text(i, value + max(counts) * .01, str(value), ha="center")
ax.set(title="Target distribution", ylabel="Sessions", xlabel="Outcome", ylim=(0, max(counts)*1.12))
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
majority_label = int(y.mode()[0])
y_baseline = np.full(len(y), majority_label)
baseline_metrics = pd.Series({
    "Accuracy": accuracy_score(y, y_baseline),
    "Precision (purchase)": precision_score(y, y_baseline, zero_division=0),
    "Recall (purchase)": recall_score(y, y_baseline, zero_division=0),
    "F1 (purchase)": f1_score(y, y_baseline, zero_division=0),
    "MCC": matthews_corrcoef(y, y_baseline)
}, name="Majority-only baseline")
display(baseline_metrics.to_frame().round(4))
print("Note: descriptive in-sample baseline, not a held-out model evaluation.")

## 4. Univariate analysis — feature types and numerical distributions
Some integer-coded columns are **categorical**, not continuous. Treat `OperatingSystems`, `Browser`, `Region`, and `TrafficType` as categories. `SpecialDay` is numeric.

In [ ]:
categorical_features = ["OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Month", "Weekend"]
X_eda = df.drop(columns=["Revenue", "Revenue_Label"], errors="ignore")
missing_expected = set(categorical_features) - set(X_eda.columns)
if missing_expected:
    raise ValueError(f"Expected categorical columns absent: {missing_expected}")
numerical_features = [c for c in X_eda.columns if c not in categorical_features]
print("Numerical (", len(numerical_features), "):", numerical_features)
print("Categorical (", len(categorical_features), "):", categorical_features)
assert len(numerical_features) == 10 and len(categorical_features) == 7
numeric_summary = X_eda[numerical_features].agg(["mean", "median", "std", "min", "max", "skew"]).T
numeric_summary["Unique Values"] = X_eda[numerical_features].nunique()
display(numeric_summary.round(3))

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 18))
for ax, column in zip(axes.flat, numerical_features):
    sns.histplot(X_eda[column], bins=30, kde=True, color="#4361ee", ax=ax)
    ax.set_title(column)
fig.suptitle("Numerical feature distributions", fontsize=16, y=1.005)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 18))
for ax, column in zip(axes.flat, numerical_features):
    sns.boxplot(x=X_eda[column], color="#b8c0ff", ax=ax)
    ax.set_title(column)
fig.suptitle("Numerical feature boxplots", fontsize=16, y=1.005)
plt.tight_layout(); plt.show()

outlier_rows = []
for column in numerical_features:
    q1, q3 = X_eda[column].quantile([.25, .75]); iqr = q3 - q1
    flagged = ((X_eda[column] < q1 - 1.5*iqr) | (X_eda[column] > q3 + 1.5*iqr))
    outlier_rows.append({"Feature": column, "IQR Flag Count": int(flagged.sum()), "IQR Flag %": round(flagged.mean()*100, 2)})
outlier_summary = pd.DataFrame(outlier_rows).sort_values("IQR Flag Count", ascending=False)
display(outlier_summary)
print("IQR flags are not automatically erroneous records.")

In [ ]:
for column in categorical_features:
    counts_col = X_eda[column].value_counts(dropna=False)
    display(pd.DataFrame({"Count": counts_col, "Percentage": (counts_col / len(X_eda) * 100).round(2)}).rename_axis(column))

fig, axes = plt.subplots(4, 2, figsize=(15, 17))
for ax, column in zip(axes.flat, categorical_features):
    order = X_eda[column].value_counts().index
    sns.countplot(data=X_eda, x=column, order=order, color="#6366f1", ax=ax)
    ax.tick_params(axis="x", rotation=45)
    ax.set_title(column)
for ax in list(axes.flat)[len(categorical_features):]:
    fig.delaxes(ax)
plt.tight_layout(); plt.show()

## 5. Bivariate analysis — predictors versus purchase
These comparisons are **descriptive associations**, not proof of causality. We compare numerical features by target class and calculate category-level purchase rates.

In [ ]:
eda = X_eda.copy()
eda["Revenue_binary"] = y.to_numpy()
fig, axes = plt.subplots(5, 2, figsize=(14, 19))
for ax, column in zip(axes.flat, numerical_features):
    sns.boxplot(data=eda, x="Revenue_binary", y=column, showfliers=False, color="#b8c0ff", ax=ax)
    ax.set_xticks([0,1], labels=["No Purchase", "Purchase"])
    ax.set_title(f"{column} by purchase outcome")
plt.tight_layout(); plt.show()

numeric_by_target = eda.groupby("Revenue_binary")[numerical_features].median().T
numeric_by_target.columns = ["No Purchase Median", "Purchase Median"]
display(numeric_by_target.round(3))

In [ ]:
for column in categorical_features:
    grouped = eda.groupby(column, dropna=False)["Revenue_binary"].agg(["size", "mean"])
    grouped.columns = ["Sessions", "Purchase Rate"]
    grouped["Purchase Rate %"] = (grouped["Purchase Rate"] * 100).round(2)
    print(f"\n{column}: purchase rate by category")
    display(grouped[["Sessions", "Purchase Rate %"]].sort_values("Sessions", ascending=False))

fig, axes = plt.subplots(4, 2, figsize=(15, 18))
for ax, column in zip(axes.flat, categorical_features):
    rates = eda.groupby(column)["Revenue_binary"].mean().sort_values(ascending=False)
    sns.barplot(x=rates.index.astype(str), y=rates.values * 100, color="#4361ee", ax=ax)
    ax.set(title=f"Purchase rate by {column}", ylabel="Purchase rate (%)", xlabel=column)
    ax.tick_params(axis="x", rotation=45)
for ax in list(axes.flat)[len(categorical_features):]:
    fig.delaxes(ax)
plt.tight_layout(); plt.show()
print("Caution: small categories may have unstable purchase-rate estimates.")

## 6. Correlation analysis
Pearson correlation is appropriate for exploring linear relationships among numeric variables. It does **not** measure all nonlinear associations. We deliberately exclude categorical integer codes from this heatmap.

In [ ]:
correlation_df = eda[numerical_features + ["Revenue_binary"]].corr(method="pearson")
plt.figure(figsize=(12, 9))
sns.heatmap(correlation_df, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=.4)
plt.title("Pearson correlations: numerical features and Revenue")
plt.tight_layout(); plt.show()

revenue_corr = correlation_df["Revenue_binary"].drop("Revenue_binary").sort_values(key=abs, ascending=False)
print("Numerical correlations with purchase outcome (ranked by absolute value):")
display(revenue_corr.to_frame("Pearson r").round(3))

## 7. EDA findings and preprocessing plan
Fill in the following interpretation after running the cells. Your actual output takes precedence over published reference statistics.

- **Data quality:** Record missing-value, duplicate-row, and conflicting-label counts. Do not claim zero duplicates unless your actual local CSV confirms it.
- **Class imbalance:** Compare your purchase share against the original ~15.47%. Accuracy alone can be misleading.
- **Numerical distributions:** Identify strongly skewed features and potential outliers. Avoid automatic deletion.
- **Categorical predictors:** Use appropriate encoding (for example, one-hot encoding) for integer-coded categories as well as text categories.
- **Scaling:** KNN, SVM, and Logistic Regression typically benefit from scaled numerical features.
- **Leakage prevention:** Split into training/test data before fitting any imputers, encoders, or scalers; use `Pipeline`/`ColumnTransformer` fitted only on training data.
- **Model comparison:** Use the same held-out test set for Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes, and SVM. Report Accuracy, AUC, Precision, Recall, F1, and MCC. Use probability or decision scores for AUC.
- **Streamlit:** Save fitted model pipelines; evaluate uploaded labeled test CSVs without retraining.

## 8. Optional preview: stratified train/test split (no model training)
The following cell demonstrates preserving class proportions. In the actual modeling step, make the split once and persist the held-out test CSV. Do not use the held-out test set to tune preprocessing or models.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_eda, y, test_size=0.20, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train purchase %:", round(y_train.mean()*100, 2))
print("Test purchase %:", round(y_test.mean()*100, 2))
print("Preview only — no model fitted and no CSV written.")

## 9. Git commit #2 — after reviewing the notebook
From the project root, update `requirements.txt` with `matplotlib`, `seaborn`, `scikit-learn`, `ipykernel` (and `jupyter` if desired), then run:

```bash
git status
git add notebooks/01_exploratory_data_analysis.ipynb requirements.txt
git diff --cached --stat
git commit -m "feat: add complete exploratory data analysis notebook"
git push origin main
```

If the notebook contains private paths, large outputs, or irrelevant metadata, clear outputs before committing. Confirm your notebook filename matches the `git add` command.